# 03 · Machine Learning Results

**Models:** the three `@champion` models in the MLflow registry:
`iiot-failure-24h` (will it fail within 24 h?), `iiot-component` (which component?) and
`iiot-anomaly` (is the sensor behaviour unusual?).

**Goal of this notebook:** show what the models do on data they never saw (the **test period,
Oct–Dec 2015**) and what that is worth, next to simple baselines.

**Before you run it:**
- Select the project's `.venv` kernel.
- Run the pipeline and the ML phase once: `iiot run` (or `iiot ml train` after `iiot run --skip-ml`).
- To browse every run, chart and model version: `mlflow ui --backend-store-uri sqlite:///mlruns/mlflow.db --workers 1`
  and open http://127.0.0.1:5000.

> The data are Microsoft's synthetic Azure Predictive Maintenance dataset. Its failures follow strong
> error and wear patterns, so the scores here are higher than a real plant would see. Costs are the
> illustrative assumptions in `config/settings.yaml`.

## 0. Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from iiot.config import get_settings
from iiot.ml.score import PREDICTIONS_FILE_NAME, load_champions
from iiot.model.warehouse import connect

settings = get_settings()
champions = load_champions()  # loads the @champion models and their model cards from MLflow
cards = {name: c.card for name, c in champions.items()}
predictions = pd.read_parquet(settings.paths.gold / PREDICTIONS_FILE_NAME)
test_period = predictions[predictions["out_of_sample"]]

# Chart style: validated colour-blind-safe categorical palette (fixed order), recessive grid.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # blue, orange, aqua, yellow
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e", "axes.titlecolor": "#0b0b0b",
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.labelsize": 9,
    "xtick.color": "#898781", "ytick.color": "#898781", "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=PALETTE), "lines.linewidth": 2,
    "legend.frameon": False, "legend.fontsize": 8, "figure.dpi": 110,
})
pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.max_colwidth", 90)
CRORE = 1e7
print({name: f"version {c.version}" for name, c in champions.items()})

## 1. The models at a glance (test period)

In [ ]:
f, c, a = cards["failure"]["test"], cards["component"], cards["anomaly"]["test"]
pd.DataFrame([
    {"model": "failure-24h", "question": "fails within 24 h?",
     "result": f"{f['events_caught']}/{f['failure_events']} failures caught, {f['false_alarms']} false alarms",
     "baseline": "error5 rule: {}/{} caught, {} false alarms".format(
         cards["failure"]["test_baselines"]["error5-rule"]["events_caught"],
         cards["failure"]["test_baselines"]["error5-rule"]["failure_events"],
         cards["failure"]["test_baselines"]["error5-rule"]["false_alarms"])},
    {"model": "component", "question": "which component?",
     "result": f"right component first {c['test']['top1_accuracy']:.0%} of the time",
     "baseline": f"most frequent component: {c['test_baseline']['top1_accuracy']:.0%}"},
    {"model": "anomaly", "question": "unusual sensors?",
     "result": f"{a['flag_rate_48h']:.1%} of rows flagged in the 48 h before a failure",
     "baseline": f"{a['flag_rate_normal']:.1%} flagged in normal operation"},
])

## 2. What the failure model is worth

Net saving = saving from failures caught early (unplanned cost − planned cost) − ₹20,000 per false alarm.

In [ ]:
options = {"never alert": cards["failure"]["test_baselines"]["never-fail"],
           "error5 rule": cards["failure"]["test_baselines"]["error5-rule"],
           "failure model": cards["failure"]["test"]}
saving = pd.Series({name: m["net_saving"] / CRORE for name, m in options.items()})
possible = cards["failure"]["test"]["potential_saving"] / CRORE

fig, ax = plt.subplots(figsize=(7, 2.8))
ax.barh(saving.index, saving.values, color=PALETTE[0], height=0.55)
for y, value in enumerate(saving.values):
    ax.annotate(f"₹{value:.2f} cr", (value, y), xytext=(4, 0), textcoords="offset points",
                va="center", color="#52514e", fontsize=8)
ax.axvline(possible, color="#898781", linestyle=":", linewidth=1)
ax.annotate(f"possible ₹{possible:.2f} cr", (possible, -0.55), ha="right", va="top", color="#898781",
            fontsize=8, annotation_clip=False)
ax.set_title("Net saving in the test period (3 months)")
ax.set_xlabel("crore INR")
ax.grid(axis="y", visible=False)
ax.grid(axis="x", color="#e1e0d9")
ax.set_xlim(0, possible * 1.15)
plt.tight_layout()
plt.show()

## 3. One machine through the test period

Predicted failure risk every 3 hours for the machine with the most test-period failures. The dotted line is
the alert threshold; dashed lines are the real failures.

In [ ]:
con = connect()
failures = con.sql("""
    SELECT machine_id, timestamp FROM fact_failure WHERE timestamp >= DATE '2015-10-01'
""").df().drop_duplicates()
machine = failures["machine_id"].value_counts().idxmax()
history = test_period[test_period["machine_id"] == machine]

fig, ax = plt.subplots(figsize=(12, 3.2))
ax.plot(history["timestamp"], history["failure_risk"], color=PALETTE[0], linewidth=1.2)
ax.axhline(cards["failure"]["threshold"], color="#898781", linestyle=":", linewidth=1)
for t in failures.loc[failures["machine_id"] == machine, "timestamp"]:
    ax.axvline(t, color="#52514e", linestyle="--", linewidth=0.8)
ax.set_title(f"Machine {machine}: predicted risk of failure within 24 h")
ax.set_ylabel("failure risk")
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 4. Why the alerts were raised (first test-period alerts)

In [ ]:
alerts = test_period[test_period["alert"]]
alerts[["machine_id", "timestamp", "failure_risk", "predicted_component", "reason_1", "reason_2"]].head(10)

## 5. Component diagnosis: actual vs diagnosed (single-component failures, test period)

In [ ]:
pd.DataFrame(cards["component"]["confusion_test"]).T.rename_axis("actually failed")

## 6. Anomaly detection: rows flagged by time before a failure (test period)

In [ ]:
buckets = {"0-24 h": "flag_rate_0_24h", "24-48 h": "flag_rate_24_48h", "48-72 h": "flag_rate_48_72h",
           "72-168 h": "flag_rate_72_168h", "normal": "flag_rate_normal"}
rates = pd.Series({k: 100 * cards["anomaly"]["test"][v] for k, v in buckets.items()})
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(rates.index, rates.values, color=PALETTE[0], width=0.6)
for x, value in enumerate(rates.values):
    ax.annotate(f"{value:.1f}%", (x, value), xytext=(0, 3), textcoords="offset points",
                ha="center", color="#52514e", fontsize=8)
ax.set_title("Rows flagged as unusual, by time before the next failure")
ax.set_ylabel("rows flagged (%)")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

## 7. Machines at risk now (latest scores, from the warehouse view `v_machine_risk_now`)

In [ ]:
con.sql("""
    SELECT risk_rank, machine_id, model, line_id, scored_at, round(failure_risk, 3) AS failure_risk,
           alert, predicted_component, anomaly_flag, reason_1
    FROM v_machine_risk_now
    ORDER BY risk_rank
    LIMIT 10
""").df()

## 8. Close the connection

Run this before rebuilding the warehouse.

In [ ]:
con.close()
print("Connection closed")